# **1. Perkenalan Dataset**


**Dataset URL**: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

**Description**: This dataset contains historical customer data from a fictional telecommunications company in California. It includes 7,043 rows and 21 features capturing customer demographics, signed services (internet, phone, security), account settings (contract type, payment method), and charges, with the target column 'Churn' indicating whether the customer left the service within the quarter.

Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [ ]:
!pip install mlflow==3.14.0 dagshub==0.7.0 skops==0.14.0 pandas==2.2.2 numpy==2.0.2 scikit-learn==1.6.1 polars


In [ ]:
print("--- Data Info ---")
df.info()

print("\n--- Data Description ---")
display(df.describe(include='all'))

print("\n--- Missing Values ---")
print(df.isnull().sum())

print("\n--- Churn Distribution ---")
print(df['Churn'].value_counts())

sns.countplot(data=df, x='Churn')
plt.title('Distribution of Target Column (Churn)')
plt.show()


# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

# **6. Model Training & Tracking**
Pada tahap ini kita melatih model dengan MLflow dan RandomizedSearchCV.

In [ ]:
# Retrieve DAGSHUB_CLIENT_TOKEN or fallback to MLFLOW_TRACKING_PASSWORD
dagshub_token = os.getenv("DAGSHUB_CLIENT_TOKEN") or os.getenv("MLFLOW_TRACKING_PASSWORD")
if dagshub_token:
    os.environ["MLFLOW_TRACKING_USERNAME"] = "lukmannurh"
    os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# Set the MLflow tracking URI headlessly
mlflow.set_tracking_uri("https://dagshub.com/lukmannurh/Eksperimen_SML_Lukman.mlflow")

# Activate autologging
mlflow.sklearn.autolog(log_models=True)

print("Starting MLflow experiment tracking session...")
with mlflow.start_run(run_name="Telco_Churn_Optimization"):
    
    # Initialize model
    rf = RandomForestClassifier(random_state=42)
    
    # Implement a controlled hyperparameter sweep
    param_dist = {
        'n_estimators': [50, 100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5, 10]
    }
    
    search = RandomizedSearchCV(
        estimator=rf, 
        param_distributions=param_dist, 
        n_iter=5, 
        cv=3, 
        scoring='accuracy',
        random_state=42,
        n_jobs=-1
    )
    
    # Train model
    search.fit(X_train, y_train)
    
    # Extract champion model
    champion_model = search.best_estimator_
    print("\nBest parameters found:", search.best_params_)
    
    # Predict on test set
    y_pred = champion_model.predict(X_test)
    
    # Calculate evaluation metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    print(f"\nAccuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    
    # Register the optimized champion model into the Model Registry
    mlflow.sklearn.log_model(
        sk_model=champion_model,
        artifact_path="champion_model_artifacts",
        registered_model_name="credit-scoring-model"
    )
    print("\nOptimized champion model successfully registered under 'credit-scoring-model'.")


# **7. Evaluation Metrics**


In [ ]:
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

print("--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred)
print(cm)

plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['No Churn (0)', 'Churn (1)'], 
            yticklabels=['No Churn (0)', 'Churn (1)'])
plt.title('Confusion Matrix')
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.show()
